In [1]:
import pandas as pd

# Path to the source Excel file
file_path = "D:\WORK - MAIN\PROJECTS\Smart Traffic Data Engineering & Analytics Pipeline\smart_traffic_dataset-v2.xlsx"

# Load both sheets
traffic_df = pd.read_excel(file_path, sheet_name="Traffic_Logs_Raw")
location_df = pd.read_excel(file_path, sheet_name="Location_Lookup")

# Check that the data was loaded correctly
print("Traffic data:")
print(traffic_df.head())

print("\nLocation lookup:")
print(location_df.head())

Traffic data:
    Record_ID           Timestamp  Location_ID Vehicle_Type  Speed_kmh  \
0  TRF-100000 2025-01-01 00:00:00          107          Car       45.8   
1  TRF-100001 2025-01-01 01:00:00          104          Car       65.4   
2  TRF-100002 2025-01-01 02:00:00          105        Truck       64.9   
3  TRF-100003 2025-01-01 03:00:00          107         car        50.1   
4  TRF-100004 2025-01-01 04:00:00          103        Truck       31.7   

   Traffic_Volume Weather_Condition   Lane_Number  
0           567.0             Clear  Express Lane  
1           372.0             clear        Lane 1  
2           169.0             foggy  Express Lane  
3           102.0             Rainy  Express Lane  
4           457.0             Foggy  Express Lane  

Location lookup:
   Location_ID           Zone_Name      Road_Type  Speed_Limit_kmh
0          101       Downtown Core     Expressway               80
1          102    Highway 401 East        Highway              100
2         

In [5]:
traffic_clean = traffic_df.copy()

In [6]:
traffic_clean["Vehicle_Type"] = traffic_clean["Vehicle_Type"].str.strip()
traffic_clean["Weather_Condition"] = traffic_clean["Weather_Condition"].str.strip()

In [7]:
traffic_clean["Vehicle_Type"] = traffic_clean["Vehicle_Type"].str.lower()
traffic_clean["Weather_Condition"] = traffic_clean["Weather_Condition"].str.lower()

In [9]:
traffic_clean["Timestamp"] = pd.to_datetime(traffic_clean["Timestamp"], errors="coerce")

In [12]:
traffic_clean = traffic_clean.dropna(subset=["Timestamp"])

In [16]:
traffic_clean = traffic_clean.dropna(subset=["Location_ID"])

In [21]:
traffic_clean["Location_ID"] = traffic_clean["Location_ID"].astype("string").str.strip().str.upper()
location_df["Location_ID"] = location_df["Location_ID"].astype("string").str.strip().str.upper()

In [22]:
invalid_location_ids = traffic_clean[
    ~traffic_clean["Location_ID"].isin(location_df["Location_ID"])
]

print("\nUnmatched Location_ID records after standardization:")
print(len(invalid_location_ids))


Unmatched Location_ID records after standardization:
0


In [23]:
traffic_clean = traffic_clean[
    traffic_clean["Location_ID"].isin(location_df["Location_ID"])
]

In [27]:
traffic_clean["Speed_kmh"] = traffic_clean["Speed_kmh"].fillna(
    traffic_clean["Speed_kmh"].median()
)

In [29]:
print("Minimum Speed:", traffic_clean["Speed_kmh"].min())
print("Maximum Speed:", traffic_clean["Speed_kmh"].max())

print("\nRows with Speed <= 0:")
print(
    traffic_clean[
        traffic_clean["Speed_kmh"] <= 0
    ][["Record_ID", "Vehicle_Type", "Location_ID", "Speed_kmh"]]
)

Minimum Speed: 5.0
Maximum Speed: 118.3

Rows with Speed <= 0:
Empty DataFrame
Columns: [Record_ID, Vehicle_Type, Location_ID, Speed_kmh]
Index: []


In [33]:
traffic_clean["Traffic_Volume"] = traffic_clean["Traffic_Volume"].fillna(
    traffic_clean["Traffic_Volume"].median()
)

In [39]:
traffic_clean["Weather_Condition"] = traffic_clean["Weather_Condition"].fillna(
    traffic_clean["Weather_Condition"].mode()[0]
)

In [42]:
print("Duplicate rows:")
print(traffic_clean.duplicated().sum())

print("\nDuplicate Record_IDs:")
print(traffic_clean["Record_ID"].duplicated().sum())

Duplicate rows:
0

Duplicate Record_IDs:
0


In [44]:
print("Negative Speed values:")
print((traffic_clean["Speed_kmh"] < 0).sum())

print("\nNegative Traffic Volume values:")
print((traffic_clean["Traffic_Volume"] < 0).sum())

Negative Speed values:
0

Negative Traffic Volume values:
0


In [48]:
print("Original rows:", len(traffic_df))
print("Cleaned rows:", len(traffic_clean))
print("Rows removed:", len(traffic_df) - len(traffic_clean))

Original rows: 12000
Cleaned rows: 12000
Rows removed: 0


In [54]:
print("Missing values in Location Lookup:")
print(location_df.isna().sum())

print("\nDuplicate Location IDs:")
print(location_df["Location_ID"].duplicated().sum())

Missing values in Location Lookup:
Location_ID        0
Zone_Name          0
Road_Type          0
Speed_Limit_kmh    0
dtype: int64

Duplicate Location IDs:
0


In [59]:
traffic_enriched = traffic_clean.merge(
    location_df,
    on="Location_ID",
    how="left"
)

In [62]:
print("Missing Zone_Name values:")
print(traffic_enriched["Zone_Name"].isna().sum())

Missing Zone_Name values:
0


In [67]:
print("Negative Speed values:")
print((traffic_enriched["Speed_kmh"] < 0).sum())

print("\nNegative Traffic Volume values:")
print((traffic_enriched["Traffic_Volume"] < 0).sum())

print("\nNegative Speed Limit values:")
print((traffic_enriched["Speed_Limit_kmh"] < 0).sum())

Negative Speed values:
0

Negative Traffic Volume values:
0

Negative Speed Limit values:
0


In [73]:
final_traffic = traffic_enriched.copy()

In [78]:
print("Missing values in final dataset:")
print(final_traffic.isna().sum())

Missing values in final dataset:
Record_ID            0
Timestamp            0
Location_ID          0
Vehicle_Type         0
Speed_kmh            0
Traffic_Volume       0
Weather_Condition    0
Lane_Number          0
Zone_Name            0
Road_Type            0
Speed_Limit_kmh      0
dtype: int64


In [80]:
final_traffic.to_csv("final_traffic_data.csv", index=False)

In [ ]:
location_df.to_csv("locations.csv", index=False)

In [ ]:
traffic_to_load = traffic_clean.copy()

In [ ]:
traffic_to_load.to_csv("traffic_logs.csv", index=False)